# Survival families

This chapter covers the Cox proportional hazards, Weibull, Log-logistic, and
Lognormal families for right-censored time-to-event data. It assumes the
general family structure of [](families.ipynb); cross-validation of survival
models is in [](model_selection.ipynb).

Survival analysis in `mboostrs` models a right-censored time-to-event
response $T$. The response enters the formula as `Surv(time, status)`, where
`status = 1` indicates an observed event and `status = 0` indicates censoring.

## Cox proportional hazards

The Cox model {cite}`cox1972regression` assumes a semi-parametric hazard

$$
h(t \mid x) = h_0(t) \exp\bigl(f(x)\bigr),
$$

where $h_0(t)$ is an unspecified baseline hazard and $f(x)$ is the
additive predictor. The partial likelihood depends only on the ordering of
event times, making the model robust to the baseline hazard's shape.
Boosting the Cox model fits $f(x)$ by maximizing the partial likelihood
component-wise {cite}`ridgeway1999`.

### The partial likelihood and its gradient

With observations $(t_i, \delta_i)$ where $\delta_i = 1$ marks an event, the
Breslow partial log-likelihood is

$$
\ell(f) = \sum_{i:\,\delta_i=1} \Bigl[ f_i - \log \Bigl(\sum_{j:\,t_j \ge t_i} w_j\, e^{f_j} \Bigr) \Bigr].
$$

The risk is $\mathcal R = -\ell$. Differentiating with respect to $f_k$
gives the negative gradient (the "martingale residual"):

$$
u_k = -\frac{\partial \mathcal R}{\partial f_k}
     = \delta_k - w_k\, e^{f_k} \sum_{i:\,\delta_i=1,\,t_i \le t_k} \frac{1}{\sum_{j:\,t_j \ge t_i} w_j\, e^{f_j}}.
$$

The first term $\delta_k$ is the observed event indicator; the second
accumulates each event's share of risk that observation $k$ contributed.
Censored observations ($\delta_k = 0$) have $u_k < 0$: they contributed risk
mass but no event.

The risk itself is

$$
\mathcal R = \sum_{i:\,\delta_i=1,\,w_i>0} \Bigl[\, \log \Bigl(\sum_{j:\,t_j \ge t_i} w_j\, e^{f_j} \Bigr) - f_i \,\Bigr].
$$

Note the $w_i > 0$ gate on the risk. In cross-validation, the rows held out
of a fold's fit carry weight zero in that fit. Such rows contribute to risk sets (their $w_j e^{f_j}$
appears in the denominator) but do not add their own partial-likelihood
terms. This matches R `mboost`'s behavior exactly.

### Worked example: a small risk set

Consider $n = 5$ observations with the following sorted event times:

| $i$ | $t_i$ | $\delta_i$ | $f_i$ | $w_i$ |
|-----|-------|------------|-------|-------|
| 1   | 1.0   | 1          | 0.5   | 1     |
| 2   | 2.0   | 1          | 0.3   | 1     |
| 3   | 3.0   | 0          | −0.2  | 1     |
| 4   | 4.0   | 1          | 0.1   | 1     |
| 5   | 5.0   | 0          | 0.0   | 1     |

The risk sets (Breslow, $\ge$) are:
- $R_1 = \{1,2,3,4,5\}$, $\quad \sum_R w_j e^{f_j} = e^{0.5} + e^{0.3} + e^{-0.2} + e^{0.1} + 1 \approx 5.922$.
- $R_2 = \{2,3,4,5\}$, $\quad \sum_R w_j e^{f_j} = e^{0.3} + e^{-0.2} + e^{0.1} + 1 \approx 4.274$.
- $R_4 = \{4,5\}$, $\quad \sum_R w_j e^{f_j} = e^{0.1} + 1 \approx 2.105$.
- Observations 3 and 5 are censored; they contribute no partial-likelihood term.

The risk is

$$
\mathcal R = \bigl[\log(5.922) - 0.5\bigr] + \bigl[\log(4.274) - 0.3\bigr] + \bigl[\log(2.105) - 0.1\bigr] \approx 3.076.
$$

The negative gradient for observation 1 (the first event):

$$
u_1 = 1 - e^{0.5}\,\frac{1}{5.922} \approx 1 - 1.649 \times 0.169 \approx 0.722.
$$

Observation 3 (censored) belongs to the risk sets of events 1 and 2, because
$t_3 \ge t_1$ and $t_3 \ge t_2$, and not to $R_4$. Its negative gradient
collects its share of those two denominators:

$$
u_3 = 0 - e^{-0.2} \cdot \Bigl(\frac{1}{5.922} + \frac{1}{4.274}\Bigr) \approx -0.819 \times 0.403 \approx -0.330.
$$

The library function `family_diagnostics` returns the negative gradient and
the risk of a family for given $y$, $f$ and $w$. Applied to the five
observations of the table:



In [1]:
import numpy as np
from mboostrs._rust import family_diagnostics

# Reproduce the worked example above: n=5, sorted event times, one censored
# pair at t=3 and t=5.
t = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
delta = np.array([1.0, 1.0, 0.0, 1.0, 0.0])
f = np.array([0.5, 0.3, -0.2, 0.1, 0.0])
w = np.ones(5)
y = np.concatenate([t, delta])  # CoxPH response is [time..., status...]

_, grad, risk = family_diagnostics("coxph", {}, y, f, w)
print(f"risk = {risk:.4f}")
print(f"u_1  = {grad[0]:.4f}  (observation 1, event)")
print(f"u_3  = {grad[2]:.4f}  (observation 3, censored)")

np.testing.assert_allclose(risk, 3.076, atol=1e-3)
np.testing.assert_allclose(grad[0], 0.722, atol=1e-3)
np.testing.assert_allclose(grad[2], -0.330, atol=1e-3)


risk = 3.0756
u_1  = 0.7216  (observation 1, event)
u_3  = -0.3298  (observation 3, censored)


`family_diagnostics` returns risk 3.0756, $u_1 = 0.7216$ and $u_3 = -0.3298$ for this table. These agree with the hand calculation in Worked example: a small risk set to the digits shown.

Censored observations contribute risk mass ($e^{f_k}$ in the denominator)
but no event term, so $u_k < 0$.

### Tie handling: Breslow

When two or more observations share the same event time, the Breslow method
treats them as a single tied group. The risk-set denominator is evaluated
once for the group, and each event in the group draws from the same
denominator. Specifically, for a tied group at time $t$ with $d$ events
and risk set $R$:

$$
\text{contribution to } \mathcal R = \sum_{k \in \text{group}} w_k \bigl[\log(\textstyle\sum_{j \in R} w_j e^{f_j}) - f_k\bigr].
$$

This is the default in R `mboost` and in `mboostrs`. The Efron method, which
removes a fraction of the tied events' own contribution from the denominator
for each successive event in the tied group, is less biased when ties are
frequent; `mboostrs` does not implement it. In practice,
with continuous survival times, ties are rare and the Breslow approximation
has negligible impact.

### Computational note: sort-once, suffix sums

A naive implementation of the risk set is an $O(n^2)$ double loop. The
engine sorts observations once by ascending time and then computes suffix
sums $\texttt{suf}[p] = \sum_{j \ge p} w_j e^{f_j}$ in reverse order
($O(n)$). Each event's denominator is a single lookup into the suffix array,
and the gradient's cumulative term is a forward scan. The result is
$O(n \log n)$ total (dominated by sorting), agreeing with the naive
computation to $\sim 10^{-15}$ {cite}`ridgeway1999`.

## Accelerated failure time models

AFT families assume $\log T = f(x) + \sigma\, \varepsilon$ with a parametric
error distribution. The predictor $f$ is on the log-time scale, and
$e^{f}$ is the time scale of the distribution. Three distributions are available:

| Family | Error $\varepsilon$ | Survival function $S(t)$ | Hazard shape |
|---|---|---|---|
| `Weibull` | minimum extreme value (Gumbel) | $\exp(-e^{\eta})$ | monotone; the AFT model is also a proportional-hazards model |
| `Loglog` (log-logistic) | logistic | $1/(1 + e^{\eta})$ | can rise and then fall; proportional-odds model |
| `Lognormal` | standard normal | $\Phi(-\eta)$ | can rise and then fall |

Here $\eta = (\log t - f)/\sigma$ is the standardized residual, used
throughout. In AFT models, a coefficient $\beta$ gives an
**acceleration factor** $e^\beta$: $\beta > 0$ means longer expected
survival, $\beta < 0$ means shorter. `predict(type="response")` returns
$e^{f}$.

Each AFT family has a nuisance scale parameter $\sigma$, re-estimated at
every iteration by minimizing the risk over $\sigma$ with Brent's method (a
one-dimensional minimizer that combines golden-section search with parabolic
interpolation) on the interval $[0, 100]$, as R's `optimize` does.

All three families follow one convention: the loss $\rho_i$ is the negative
log-likelihood of observation $i$ (up to a constant), with an event
contributing $-\log$ of the density and a censored observation
$-\log S(t_i)$; the negative gradient is $u_i = -\partial\rho_i/\partial f_i$.
Because $\partial\eta_i/\partial f_i = -1/\sigma$,
$u_i = \sigma^{-1}\,\partial\rho_i/\partial\eta_i$. A positive $u_i$ moves
$f_i$ up, towards longer predicted times.

### Weibull: loss and negative gradient

$$
\rho_i = \delta_i \bigl(e^{\eta_i} - \eta_i + \log\sigma\bigr) + (1 - \delta_i)\, e^{\eta_i},
$$

$$
u_i = \frac{1}{\sigma}\Bigl[\delta_i\bigl(e^{\eta_i} - 1\bigr) + (1 - \delta_i)\, e^{\eta_i}\Bigr].
$$

$e^{\eta_i} = (t_i/e^{f_i})^{1/\sigma}$ is the cumulative hazard at $t_i$.
For an event ($\delta_i = 1$), $u_i$ is zero at $f_i = \log t_i$, positive
when $f_i < \log t_i$ and negative when $f_i > \log t_i$: the step moves
$f_i$ towards $\log t_i$. For a censored observation ($\delta_i = 0$),
$u_i = e^{\eta_i}/\sigma > 0$ always, so the step moves $f_i$ up, because
the survival time is known to exceed $t_i$.

The risk is $\mathcal R = \sum_i w_i\,\rho_i$, for this and the other two
families.

### Log-logistic: loss and negative gradient

With $\text{nom}_i = e^{-\eta_i} + 1$, so that
$\log(1 + e^{\eta_i}) = \eta_i + \log\text{nom}_i$:

$$
\rho_i = \delta_i\bigl(\eta_i + 2\log\,\text{nom}_i + \log\sigma\bigr) + (1 - \delta_i)\bigl(\eta_i + \log\,\text{nom}_i\bigr),
$$

$$
u_i = \frac{1}{\sigma}\Bigl[\delta_i\Bigl(\frac{2}{\text{nom}_i} - 1\Bigr) + \frac{1 - \delta_i}{\text{nom}_i}\Bigr].
$$

The censored term is positive, as for the Weibull family. The event term is
zero at $\eta_i = 0$, the median of the logistic error.

### Lognormal: loss and negative gradient

$$
\rho_i = \delta_i\Bigl(\frac{\eta_i^2}{2} + \log\sigma\Bigr) - (1 - \delta_i)\,\log\Phi(-\eta_i),
$$

$$
u_i = \delta_i\,\frac{\eta_i}{\sigma} + (1 - \delta_i)\,\frac{\phi(\eta_i)}{\sigma\,\Phi(-\eta_i)},
$$

where $\Phi$ and $\phi$ are the standard normal CDF and density. The
censored term is the inverse Mills ratio $\phi(\eta)/\Phi(-\eta)$ divided by
$\sigma$; it is positive.

### Calibration of $\sigma$ and the offset

All three AFT families calibrate $\sigma$ by Brent minimization of the risk
over $[0, 100]$, with R's `optimize` tolerance
(`.Machine$double.eps^0.25`, about $1.2 \times 10^{-4}$). The offset
$\hat f^{[0]}$ is the Brent minimizer of the constant-fit risk at the
starting value $\sigma = 1$ over $[0, \max_i \log t_i]$ (rows with positive
weight), as in R. The bracket starts at 0, so the offset is never negative:
when every observed time is below 1, the offset is 0.

## Numerical check: AFT parameter recovery

In [2]:
import numpy as np
import mboostrs as mb

# A Weibull AFT model: log T = f(x) + sigma * eps, eps standard extreme-value.
# Generate this directly as T = exp(f(x)) * Exponential(1) (a Weibull with
# shape 1/sigma=1 and scale exp(f(x))), so the fitted linear predictor should
# recover the true coefficient of x on the log-time scale.
rng = np.random.default_rng(1)
n = 1000
true_beta = 1.5
x = rng.normal(size=n)
f_true = true_beta * x
event_time = np.exp(f_true) * rng.exponential(1.0, size=n)
censor_time = rng.exponential(3.0, size=n)
time = np.minimum(event_time, censor_time)
status = (event_time <= censor_time).astype(float)
print(f"censoring rate: {1 - status.mean():.2f}")

data = {"x": x, "time": time, "status": status}
fit = mb.glmboost(
    "Surv(time, status) ~ bols(x)", data=data, family=mb.Weibull(),
    control=mb.boost_control(mstop=1000, nu=0.1),
)
coef_x = fit.coef()["x"]
print(f"recovered coef(x) = {coef_x:.4f}  (true = {true_beta})")

assert abs(coef_x - true_beta) < 0.1


censoring rate: 0.30
recovered coef(x) = 1.5369  (true = 1.5)


The question is whether boosting with the `Weibull` family recovers a known
AFT coefficient. The simulated data have $n = 1000$ rows,
$\log T = 1.5\,x + \varepsilon$ with $x$ standard normal and $\varepsilon$
standard minimum extreme value ($\sigma = 1$), and independent exponential
censoring times with mean 3, which censor
{eval}`f"{100 * (1 - status.mean()):.0f}"`% of the rows. The model is a
single unpenalized linear learner, `bols(x)`, run for 1000 iterations at
$\nu = 0.1$. The fitted coefficient is {eval}`round(coef_x, 3)` against the
true 1.5. For a user, this means the fitted coefficient of a Weibull model is
on the log-time scale and has the AFT interpretation of the table above.

## Response format

The survival response is written inside the formula string as
`Surv(time_column, status_column)`; the formula parser reads the two named
columns from the data dict or DataFrame, which must contain both. A
two-column array passed as the response is rejected, and Python has no
`Surv` object to construct.

## Time-dependent effects

A Cox coefficient that changes with analysis time requires start-stop
risk-set data and a likelihood evaluated at each risk-set time.
`bbs(time, by=covariate)` instead uses each subject's observed event or
censoring time as an ordinary predictor. That value is determined during
follow-up and cannot diagnose proportional hazards. Use a survival
implementation with time-dependent covariates for a hazard ratio that changes
with time.

## Limitations

- AIC is not available for survival families (no dispersion parameter).
- For `CoxPH`, `predict()` returns the linear predictor $\hat f(x)$ (log
  hazard ratio). The Breslow estimate of the cumulative baseline hazard
  $H_0(t)$ is computed once after fitting and stored with the model;
  `survfit()` returns the baseline survival curve (or one curve per row of
  new data) and `survival_at(times, newdata)` returns $S(t \mid x)$ at fixed
  times. Both require unit case weights. See [](../usage/survival.ipynb) for
  a worked example with cross-validated `mstop` selection.
- All survival families require at least one observed event; a data set with
  all censored observations raises an error.

## Summary

- The Cox PH family uses the Breslow partial likelihood. The negative gradient
  (martingale residual) is $u_k = \delta_k - w_k e^{f_k} \sum 1/\text{denom}$;
  the risk sums log-denominator minus $f_k$ over events only.
- All three AFT families share the pattern: standardized residual
  $\eta_i = (\log t_i - f_i)/\sigma$, piecewise loss (events vs censored),
  and closed-form ngradient. $\sigma$ is calibrated per iteration by Brent
  minimization.
- Tie handling uses the Breslow method (grouped denominator). Efron
  approximation is not implemented.
- The response is a right-censored pair $(t_i, \delta_i)$ expressed as
  `Surv(time, status)` in the formula. Start-stop data and time-dependent
  coefficients are not implemented.
- In all three AFT families the loss is the negative log-likelihood and the
  negative gradient is $-\partial\rho/\partial f$; censored observations
  always have a positive negative gradient.

## See also

- [](../usage/survival) — running examples with data generation, cvrisk
  tuning, and diagnostics.
- [](families.ipynb) — the family catalogue and how to add a new one.
- [](gamlss.ipynb) — distributional survival models (WeibullLSS, LogLogLSS, …)
  in the GAMLSS framework.
- [](../implementation/parity_and_bugs) — documented differences in
  `gamboostLSS` survival families (LogLogLSS σ offset, WeibullLSS μ offset).